# Athena 01 — Create the `yelp_db` Database

This notebook creates the Glue/Athena database that will hold metadata for the raw Yelp review data uploaded by notebook `01_setup_S3_bucket.ipynb`.

Follows the same pattern as the Heart Valve `01_Create_Athena_Database.ipynb`.

In [2]:
!pip install --disable-pip-version-check --quiet awswrangler PyAthena pandas

In [3]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

%store -r bucket
%store -r region
%store -r athena_staging_prefix

if "bucket" not in dir() or not bucket:
    account_id = boto3.client("sts").get_caller_identity()["Account"]
    bucket = f"yelp-sentiment-mlops-{account_id}"
    region = boto3.Session().region_name
    athena_staging_prefix = "athena/staging"

database_name = "yelp_db"
s3_staging_dir = f"s3://{bucket}/{athena_staging_prefix}/"

print("Bucket:           ", bucket)
print("Region:           ", region)
print("Athena staging:   ", s3_staging_dir)
print("Database to create:", database_name)

%store database_name
%store s3_staging_dir

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Bucket:            yelp-sentiment-mlops-045456814877
Region:            us-east-1
Athena staging:    s3://yelp-sentiment-mlops-045456814877/athena/staging/
Database to create: yelp_db
Stored 'database_name' (str)
Stored 's3_staging_dir' (str)


In [4]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
print(statement)
pd.read_sql(statement, conn)

CREATE DATABASE IF NOT EXISTS yelp_db


/tmp/ipykernel_26058/1132371316.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [5]:
df_show = pd.read_sql("SHOW DATABASES", conn)
df_show

/tmp/ipykernel_26058/1694533727.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_show = pd.read_sql("SHOW DATABASES", conn)


,database_name
0,default
1,dsoaws
2,sagemaker_featurestore
3,yelp_db


In [6]:
ingest_create_athena_db_passed = database_name in df_show.values
%store ingest_create_athena_db_passed
print("Create Athena DB passed:", ingest_create_athena_db_passed)

Stored 'ingest_create_athena_db_passed' (bool)
Create Athena DB passed: True


## Done

Continue to `02_Register_S3_With_Athena.ipynb` to register the raw-reviews CSV as a queryable Athena table.